# Model Evaluation With MLflow

This notebook mirrors the evaluation stage and shows how to log metrics and the model to MLflow / DagsHub using the same environment-variable style as the reference project.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
import yaml
import mlflow
import mlflow.keras
import tensorflow as tf

In [ ]:
with open("params.yaml", "r", encoding="utf-8") as file_obj:
    params = yaml.safe_load(file_obj)

tracking_uri = os.getenv(
    "MLFLOW_TRACKING_URI",
    "https://dagshub.com/Lokesh-7368/Predictive-Analysis-Project.mlflow"
)
tracking_uri

In [ ]:
datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20
)

valid_generator = datagenerator.flow_from_directory(
    directory="artifacts/data_ingestion/Chest-CT-Scan-data",
    subset="validation",
    shuffle=False,
    target_size=tuple(params["IMAGE_SIZE"][:-1]),
    batch_size=params["BATCH_SIZE"],
    interpolation="bilinear",
    class_mode="categorical"
)

model = tf.keras.models.load_model("artifacts/training/model.h5")
score = model.evaluate(valid_generator, verbose=0)
score

In [ ]:
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

with mlflow.start_run(run_name="vgg16-chest-ct-evaluation"):
    mlflow.log_params(params)
    mlflow.log_metrics({"loss": score[0], "accuracy": score[1]})
    if tracking_url_type_store != "file":
        mlflow.keras.log_model(model, "model", registered_model_name="VGG16Model")
    else:
        mlflow.keras.log_model(model, "model")

print("MLflow logging completed")